# Phase 2: Factor Construction

## Factors (FF 2015 definitions, Indian adaptation)

| Factor | Sort variable | Definition (formation year t) |
|---|---|---|
| MKT | — | VW return of all universe members − Rf (weights = ME at end of month t−1) |
| SMB | Size | ME at end of June t |
| HML | B/M | BE / ME, both at the firm's latest fiscal year-end public by June t |
| RMW | OP | (Operating profit before depreciation − Interest) / BE, same fiscal year |
| CMA | Inv | TA_FY / TA_FY−1 − 1 (12-month gap, same statement basis) |

OP follows FF: revenue − COGS − SG&A − interest, divided by book equity
(other income excluded). Inv uses the lagged total-assets denominator.

## Universe
Point-in-time Nifty-500 equivalent from Phase 1: each June t, the top 500 NSE stocks by full
market cap (latest NSE ranking file on or before 31 March t). Only rows with `in_universe == True`
are used, i.e. the holding year July t → June t+1 of each June the stock was a member.

## Timing
- Portfolios formed at **end of June t**, held **July t → June t+1**.
- Lag: fundamentals are treated as public 3 months after the fiscal year-end (SEBI LODR Reg 33
  allows 60 days for audited annual results), so March-t data is usable in June t
  (FF use ≥ 6 months for US 10-K filings).

## Breakpoints
- Size: median June ME of all universe members with a valid June ME (one breakpoint per year,
  shared by all three sorts).
- Characteristics: 30th/70th percentiles of **Big stocks only**, applied to all stocks
  (FF international convention, the analog of NYSE breakpoints).

## Eligibility (per sort)
- All: universe member, valid June ME > 0, traded in the 12 months before June t
- HML: BE > 0, valid ME at fiscal year-end
- RMW: BE > 0, valid OP, non-financial firm
- CMA: valid asset growth (12-month change, same basis)

## Weighting
- Value-weighted with ME at end of previous month.
- A portfolio-month with fewer than MIN_STOCKS stocks is set to NaN
  (documented deviation; FF impose no minimum because the US universe is large).

## Known limitations
- Universe applies the Nifty 500's size rule only (not its liquidity/F&O rules or March/September reviews).
- "Small" = the smaller half of the top 500, i.e. mostly mid- and small-caps; SMB ≈ mid/small minus large.
- Weights use full market cap, not free float.
- Delisting returns exist only for mergers into a listed stock.
- Rf is the FRED 3-month interbank rate (proxy for the 91-day T-bill).
- Independent sorts (FF convention) CAN produce thin cells; they do not prevent them.

In [1]:
# ==========================================================
# STEP 0: Imports + Paths
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

# Methodology constants
BP_LOW = 0.30  # characteristic low breakpoint
BP_HIGH = 0.70  # characteristic high breakpoint
FORM_MONTH = 6  # June snapshot
MIN_STOCKS = 5  # min stocks per portfolio-month (documented deviation)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

In [2]:
# ==========================================================
# STEP 1: Load Phase 1 Outputs
# ==========================================================

panel_all = pd.read_csv(PROCESSED_DIR / "monthly_panel.csv", parse_dates=["date", "formation_date"])

# Point-in-time universe: a stock-month counts only in the holding year of a June it was a member
final = panel_all[panel_all["in_universe"]].copy()

chars = pd.read_csv(PROCESSED_DIR / "characteristics_annual.csv")   # all rows are universe members

rf = pd.read_csv(PROCESSED_DIR / "rf_monthly.csv", parse_dates=["date"])

print("full panel:", panel_all.shape, "| universe stock-months:", final.shape)
print(final.columns.tolist())

print("\nchars:", chars.shape)
print(chars.columns.tolist())

full panel: (126202, 16) | universe stock-months: (61677, 16)
['symbol', 'date', 'ret', 'close', 'shares', 'me', 'me_lag', 'shares_source', 'trade_days', 'tt_days', 'Rf_monthly', 'excess_ret', 'formation_date', 'in_universe', 'is_financial', 'broad_sector']

chars: (5500, 32)
['formation_date', 'symbol', 'nse_rank', 'fy_end', 'year', 'sales', 'operating_profit', 'other_income', 'interest_expense', 'depreciation', 'net_income', 'total_assets', 'equity_capital', 'book_value', 'basis', 'operating_profitability', 'months_since_prev_fy', 'prev_basis', 'check_me', 'available_date', 'prev_total_assets', 'asset_growth', 'me_fy_end', 'bm', 'is_financial', 'broad_sector', 'me_formation', 'shares_source', 'months_traded', 'tv_median_12m', 'trade_days_12m', 'in_universe']


In [3]:
final.columns

Index(['symbol', 'date', 'ret', 'close', 'shares', 'me', 'me_lag',
       'shares_source', 'trade_days', 'tt_days', 'Rf_monthly', 'excess_ret',
       'formation_date', 'in_universe', 'is_financial', 'broad_sector'],
      dtype='object')

In [4]:
rf.head()

,date,Rf_monthly
0,2011-11-30,NaN
1,2011-12-31,0.007391
2,2012-01-31,0.007128
3,2012-02-29,0.007179
4,2012-03-31,0.007432


In [5]:
rf["Rf_monthly"].describe(percentiles=[0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

count    178.000000
mean       0.005332
std        0.001385
min        0.002546
1%         0.002656
10%        0.002943
25%        0.004506
50%        0.005393
75%        0.006187
90%        0.007144
99%        0.007873
max        0.009445
Name: Rf_monthly, dtype: float64

In [6]:
# ==========================================================
# STEP 2: Data Checks on Phase 1 Outputs
# ==========================================================

# 2.1 header and date parsing

# CSV reload turns dates into text -> convert back
for col in ["fy_end", "available_date", "formation_date"]:
    chars[col] = pd.to_datetime(chars[col])

final["formation_date"] = pd.to_datetime(final["formation_date"])

In [7]:
# 2.2 keeping only those columns which are used in phase 2

cols_needed = [
    "symbol",
    "date",
    "ret",
    "me",
    "me_lag",  # weight for month t = ME at end of t-1
    "Rf_monthly",
    "is_financial",
    "formation_date",  # links each month to its June formation
]

final = final[cols_needed].copy()

In [8]:
# 2.3 basic cleaning

# inf can appear from divisions in Phase 1 (e.g., zero prior assets)
final = final.replace([np.inf, -np.inf], np.nan)
chars = chars.replace([np.inf, -np.inf], np.nan)

# Sort by stock, then time (needed for the me_lag check)
final = final.sort_values(["symbol", "date"]).reset_index(drop=True)

In [9]:
# 2.4 uniqueness/deduplication

dup_final = final.duplicated(["symbol", "date"]).sum()
dup_chars = chars.duplicated(["symbol", "formation_date"]).sum()

print("Duplicate stock-months :", dup_final)  # must be 0
print("Duplicate stock-years  :", dup_chars)  # must be 0

Duplicate stock-months : 0
Duplicate stock-years  : 0


In [10]:
# 2.5 missing values overview

print(final.isnull().sum())

print()
char_cols = [
    "me_formation",
    "book_value",
    "bm",
    "operating_profitability",
    "asset_growth",
]
print(chars[char_cols].isnull().sum())

symbol              0
date                0
ret                 0
me                 43
me_lag             43
Rf_monthly        498
is_financial        0
formation_date      0
dtype: int64

me_formation                12
book_value                 166
bm                         229
operating_profitability    819
asset_growth               277
dtype: int64


In [11]:
# 2.6 risk-free rate checks

print(final["Rf_monthly"].describe())

# Months with no Rf -> MKT cannot be computed there
missing_rf = final.loc[final["Rf_monthly"].isna(), "date"]
print(missing_rf.drop_duplicates().sort_values())

# Suspiciously low values (Rf proxy should never be ~0%)
print(final.nsmallest(5, "Rf_monthly")[["date", "Rf_monthly"]])

count    61179.000000
mean         0.004666
std          0.001028
min          0.002546
25%          0.004229
50%          0.005090
75%          0.005461
max          0.005860
Name: Rf_monthly, dtype: float64
75   2026-10-31
Name: date, dtype: datetime64[ns]
          date  Rf_monthly
5   2020-12-31    0.002546
129 2020-12-31    0.002546
221 2020-12-31    0.002546
293 2020-12-31    0.002546
381 2020-12-31    0.002546


In [12]:
# 2.7 verifying phase 1's ME lag (on the full panel: the universe filter leaves gaps between member years)

check_lag = panel_all.sort_values(["symbol", "date"])
me_prev = check_lag.groupby("symbol")["me"].shift(1)

diff = (me_prev - check_lag["me_lag"]).abs()
print(diff.describe())  # max should be ~0

print("me_lag missing (universe rows):", final["me_lag"].isna().sum())

count    125116.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
dtype: float64
me_lag missing (universe rows): 43


In [13]:
# 2.8 drop missing returns and check extremes

final = final.dropna(subset=["ret"])

# Phase 1 returns already include dividends and every split/bonus/rights/demerger/merger adjustment,
# so extremes here should be real events (crashes, re-ratings), not data errors
extreme = final[(final["ret"] < -0.5) | (final["ret"] > 1.0)]

print("Extreme returns:", len(extreme))
print(extreme[["date", "symbol", "ret"]].sort_values("ret"))

Extreme returns: 93
            date      symbol       ret
26322 2016-11-30        ICIL -0.810570
33470 2016-12-31      KNRCON -0.768059
10603 2018-09-30    CCAVENUE -0.743756
19308 2020-03-31     FRETAIL -0.742052
19344 2020-03-31         FSC -0.736849
...          ...         ...       ...
47538 2020-06-30    RELINFRA  1.310448
25097 2021-02-28  HINDCOPPER  1.491272
18245 2022-12-31        FACT  1.558068
53136 2020-02-29       SUVEN  1.653105
46884 2017-12-31        RCOM  1.869048

[93 rows x 3 columns]


No split/bonus correction in Phase 2: Phase 1 adjusts returns for every corporate action (Steps 5–6, checked in Step 5.9), so the remaining extreme months are real price moves and are kept (FF do not trim returns).

In [14]:
# 2.9 annual FY data check

# Mostly month 3 (March FY); a few Dec/Sep MNC year-ends are fine
print(chars["fy_end"].dt.month.value_counts())


fy_end
3.0     5169
12.0     131
6.0       23
9.0       11
Name: count, dtype: int64


| FY ends in | Rows | Who these usually are |
|---|---|---|
| March | 5,433 (97%) | Standard Indian April–March year |
| December | 142 | MNC subsidiaries following their parent's calendar year |
| September | 13 | e.g., Siemens-type parents |
| June | 9 | Rare, or a company in transition |
| February | 1 | Odd, probably a one-off transition period or a data error |

Lags will be different for different FY endings, for ex: 6 for FY ending in Dec.

In [15]:
# 2.10 look-ahead guard

# Accounting data must be public before the June formation date
late = chars["available_date"] > chars["formation_date"]
print("Look-ahead rows:", late.sum())  # must be 0

# Months between FY end and formation
year_gap = chars["formation_date"].dt.year - chars["fy_end"].dt.year
month_gap = chars["formation_date"].dt.month - chars["fy_end"].dt.month
lag_months = year_gap * 12 + month_gap

print(lag_months.value_counts().sort_index())

Look-ahead rows: 0
3.0     5169
6.0      131
9.0       11
12.0      23
Name: count, dtype: int64


In [16]:
# Valid: FY ended 3 to 14 months before June formation

stale = (lag_months < 3) | (lag_months > 14)
print("Stale or too-recent rows:", stale.sum())

print(chars.loc[stale, ["symbol", "fy_end", "formation_date"]])

chars = chars[~stale].copy()

Stale or too-recent rows: 0
Empty DataFrame
Columns: [symbol, fy_end, formation_date]
Index: []


In [17]:
# 2.11 OP formula check

# FF: OP = (Revenue - COGS - SG&A - Interest) / Book Equity
op_ff = chars["operating_profit"] - chars["interest_expense"]
op_ff = op_ff / chars["book_value"]

gap = (op_ff - chars["operating_profitability"]).abs()
print(gap.describe())  # max should be ~0

count    4.681000e+03
mean     3.286863e-17
std      5.953753e-17
min      0.000000e+00
25%      0.000000e+00
50%      2.775558e-17
75%      5.551115e-17
max      3.552714e-15
dtype: float64


In [18]:
# 2.12 B/M check

# FF: B/M = BE / ME at fiscal year end
bm_ff = chars["book_value"] / chars["me_fy_end"]

gap_bm = (bm_ff - chars["bm"]).abs()
print(gap_bm.describe())  # max should be ~0

count    5.271000e+03
mean     3.693034e-17
std      6.155740e-17
min      0.000000e+00
25%      0.000000e+00
50%      2.775558e-17
75%      5.551115e-17
max      8.881784e-16
dtype: float64


In [19]:
# 2.13 Inv consistency

period = chars["months_since_prev_fy"]

first_obs = chars["prev_basis"].isna()  # no prior FY
switch = ~first_obs & (chars["basis"] != chars["prev_basis"])
data_gap = period.isin([24, 36])  # missing year(s)
transition = period.notna() & (period % 12 != 0)  # 9, 15 months

print("First observations :", first_obs.sum())
print("Basis switches     :", switch.sum())
print("Data gaps (24/36)  :", data_gap.sum())
print("Transition (9/15)  :", transition.sum())

First observations : 169
Basis switches     : 83
Data gaps (24/36)  : 0
Transition (9/15)  : 24


In [20]:
# Inv: needs a clean 12-month change on the same basis
bad_inv = switch | data_gap | transition
had_inv = chars["asset_growth"].notna()

print("Asset growth actually lost:", (bad_inv & had_inv).sum())
chars.loc[bad_inv, "asset_growth"] = np.nan

# OP: only transition periods are non-annual
had_op = chars["operating_profitability"].notna()
print("OP actually lost:", (transition & had_op).sum())
chars.loc[transition, "operating_profitability"] = np.nan

Asset growth actually lost: 0
OP actually lost: 23


In [21]:
print(chars["basis"].value_counts(dropna=False))
print(chars.loc[switch, ["basis", "prev_basis"]].value_counts())

basis
consolidated    4717
standalone       617
NaN              166
Name: count, dtype: int64
basis         prev_basis  
consolidated  standalone      69
standalone    consolidated    14
Name: count, dtype: int64


**FF timing convention (Indian adaptation):**

- Portfolios formed at **end of June t** (`formation_date` = 30 June t)
- Accounting data: the latest fiscal year public by June t (3-month reporting lag; March year-ends
  for most firms, so FY ending March t). SEBI LODR allows 60 days, so there is no look-ahead.
  (FF use a ≥6-month lag for US 10-K filings.)
- Size: ME at end of June t. B/M: BE / ME, both at the fiscal year-end.
- Portfolios held **July t → June t+1**, rebalanced annually
- Monthly value weights use ME at end of the previous month (`me_lag`)

In [22]:
# ==========================================================
# STEP 3: FF Annual Formation Dataset
# ==========================================================

**What this step does:** prepares one row per stock per June formation date,
holding everything the 2×3 sorts need.

**FF timing (Indian adaptation):**

| Item | Value used | FF original (US) |
|---|---|---|
| Formation date | 30 June t | 30 June t |
| Size | ME at end of June t | ME at end of June t |
| B/M | BE / ME, both at the latest FY end public by June t (mostly March t) | BE (FY in t−1) / ME end of Dec t−1 |
| OP, Inv | FY ending ≤ March t (12-month periods only) | FY ending in t−1 |
| Holding period | July t → June t+1 | July t → June t+1 |

**Source:** `characteristics_annual.csv` from Phase 1, already checked in Step 2 for
look-ahead (2.10), formulas (2.11–2.12) and period consistency (2.13).

**Link to monthly returns:** every month in `final` carries a `formation_date`;
Jul t – Jun t+1 → 30 June t.

In [23]:
# 3.1 Check Phase 1's month -> formation_date mapping

year = final["date"].dt.year
month = final["date"].dt.month

# FF rule: Jul-Dec -> same year, Jan-Jun -> previous year
expected_year = np.where(month >= 7, year, year - 1)

wrong_year = final["formation_date"].dt.year != expected_year
wrong_month = final["formation_date"].dt.month != 6

print("Rows with wrong formation year :", wrong_year.sum())  # must be 0
print("Rows with non-June formation   :", wrong_month.sum())  # must be 0

# Visual check around one boundary (June -> July 2019)
boundary = final[final["date"].between("2019-05-31", "2019-08-31")]
print(boundary[["date", "formation_date"]].drop_duplicates())

Rows with wrong formation year : 0
Rows with non-June formation   : 0
          date formation_date
110 2019-05-31     2018-06-30
111 2019-06-30     2018-06-30
112 2019-07-31     2019-06-30
113 2019-08-31     2019-06-30


Results for respective months look correct

In [24]:
# 3.2 Formation snapshot: one row per stock per June

keep_cols = [
    "symbol",
    "formation_date",
    "me_formation",  # size: ME at end of June t
    "book_value",  # for the positive-BE filter
    "bm",  # HML sort variable
    "operating_profitability",  # RMW sort variable
    "asset_growth",  # CMA sort variable
    "is_financial",
    "check_me",  # Phase 1 flag: BE > 0 but < 1% of total assets
    "trade_days_12m",  # 0 = no trading -> stale June price
]

formation = chars[keep_cols].copy()

formation = formation.rename(
    columns={
        "me_formation": "me_june",
        "operating_profitability": "op",
        "asset_growth": "inv",
    }
)

print(formation.shape)
print(formation.head())

(5500, 10)
     symbol formation_date        me_june  book_value        bm        op  \
0       TCS     2016-06-30  502960.700000     71072.0  0.143146  0.431168   
1  RELIANCE     2016-06-30  314241.072600    231556.0  0.683692  0.164496   
2      INFY     2016-06-30  267879.040000     61744.0  0.221505  0.276610   
3  HDFCBANK     2016-06-30  298041.254775     74304.0  0.274443       NaN   
4       ITC     2016-06-30  296683.940400     42680.0  0.161638  0.338636   

        inv  is_financial check_me  trade_days_12m  
0  0.204220         False    False           248.0  
1  0.187341         False    False           248.0  
2  0.132888         False    False           248.0  
3  0.255503          True    False           248.0  
4  0.124021         False    False           248.0  


In [25]:
# 3.3 Does chars' June ME match the panel's June ME?

june = panel_all[panel_all["date"].dt.month == 6][["symbol", "date", "me"]]
june = june.rename(columns={"date": "formation_date", "me": "me_june_panel"})

keys = ["symbol", "formation_date"]
check = formation.merge(june, on=keys, how="left")

ratio = check["me_june"] / check["me_june_panel"]
print(ratio.describe())

off = (ratio - 1).abs() > 0.01
print("Rows where June ME differs by > 1%:", off.sum())
print(check.loc[off, keys + ["me_june", "me_june_panel"]])

count    5488.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64
Rows where June ME differs by > 1%: 0
Empty DataFrame
Columns: [symbol, formation_date, me_june, me_june_panel]
Index: []


In [26]:
# 3.4 Uniqueness and coverage per year

dups = formation.duplicated(["symbol", "formation_date"]).sum()
print("Duplicate stock-years:", dups)  # must be 0

has = pd.DataFrame(
    {
        "formation_date": formation["formation_date"],
        "stocks": 1,
        "me_june": formation["me_june"] > 0,
        "bm": formation["bm"] > 0,
        "op": formation["op"].notna(),
        "inv": formation["inv"].notna(),
    }
)

coverage = has.groupby("formation_date").sum()
print(coverage)

Duplicate stock-years: 0
                stocks  me_june   bm   op  inv
formation_date                                
2016-06-30         500      498  469  413  444
2017-06-30         500      496  468  420  462
2018-06-30         500      499  472  422  466
2019-06-30         500      500  471  417  469
2020-06-30         500      499  475  422  477
2021-06-30         500      499  474  414  473
2022-06-30         500      499  479  426  485
2023-06-30         500      500  486  427  481
2024-06-30         500      500  490  431  489
2025-06-30         500      499  491  432  487
2026-06-30         500      499  496  434  490


In [27]:
# 3.5 Share of monthly returns that can enter factor portfolios

keys = ["symbol", "formation_date"]

linked = final[keys + ["date"]].merge(
    formation[keys], on=keys, how="left", indicator=True
)

linked["in_formation"] = linked["_merge"] == "both"

by_year = linked.groupby(linked["date"].dt.year)["in_formation"].mean()
print(by_year.round(2))

date
2016    1.0
2017    1.0
2018    1.0
2019    1.0
2020    1.0
2021    1.0
2022    1.0
2023    1.0
2024    1.0
2025    1.0
2026    1.0
Name: in_formation, dtype: float64


The point-in-time universe starts at the June 2016 formation (first NSE ranking file with FY2015 fundamentals for the prior-year asset growth), so there are no sparse early years to remove.

In [28]:
SAMPLE_START = "2016-06-30"

before = len(formation)
formation = formation[formation["formation_date"] >= SAMPLE_START].copy()

print("Pre-2015 formation rows removed:", before - len(formation))

Pre-2015 formation rows removed: 0


In [29]:
# 3.6 Financials: kept in all sorts except RMW

is_fin = formation["is_financial"].fillna(False).astype(bool)

# How much data do financials have for each sort variable?
print(formation.loc[is_fin, ["bm", "op", "inv"]].notna().sum())

# OP is not comparable for banks/NBFCs -> drop them from RMW only
formation.loc[is_fin, "op"] = np.nan

print("Financial stock-years kept   :", is_fin.sum())
print("Excluded from RMW sort only  :", is_fin.sum())

bm     901
op     310
inv    884
dtype: int64
Financial stock-years kept   : 902
Excluded from RMW sort only  : 902


**Treatment of financial firms**

- **Kept** in MKT, SMB, HML and CMA, following FF (2015), which includes all common stocks.
- **Excluded from RMW only.** For non-financial firms, OP = (Revenue − COGS − SG&A − Interest) / BE,
  where interest is a financing cost deducted after operations. For banks and NBFCs, interest is the
  core cost of doing business (their "raw material") and there is no COGS, so the same formula
  is not economically comparable.
- **Robustness check:** factors will also be rebuilt without financials and compared.

In [30]:
# 3.7a What are these Phase 1 columns? (inspect, don't filter yet)

flag_cols = ["check_me", "months_traded", "tv_median_12m", "trade_days_12m"]

print(chars[flag_cols].describe(include="all"))
print(chars["check_me"].value_counts(dropna=False).head())

       check_me  months_traded  tv_median_12m  trade_days_12m
count      5334    5493.000000   5.304000e+03     5304.000000
unique        2            NaN            NaN             NaN
top       False            NaN            NaN             NaN
freq       5330            NaN            NaN             NaN
mean        NaN     101.831968   1.907229e+10      246.555241
std         NaN      48.781000   3.795841e+10        2.632920
min         NaN       4.000000   2.982261e+06      169.000000
25%         NaN      66.000000   1.518150e+09      244.000000
50%         NaN     102.000000   6.199395e+09      247.000000
75%         NaN     138.000000   2.038693e+10      249.000000
max         NaN     186.000000   6.698988e+11      250.000000
check_me
False    5330
NaN       166
True        4
Name: count, dtype: int64


In [31]:
# 3.7b FF requirement: a valid June ME and some trading in the past 12 months

no_me = ~(formation["me_june"] > 0)              # NaN or zero ME
no_trade = formation["trade_days_12m"] == 0      # stale June price
invalid = no_me | no_trade

print("Removed (no valid June ME or no trading):", invalid.sum())
formation = formation[~invalid].copy()

# check_me = BE > 0 but below 1% of total assets -> extreme B/M and OP.
# FF only require BE > 0, so these rows stay in; listed for information.
tiny_be = formation["check_me"].fillna(False).astype(bool)
print("Tiny-BE rows kept:", tiny_be.sum())
print(formation.loc[tiny_be, ["symbol", "formation_date", "book_value", "bm", "op"]])

Removed (no valid June ME or no trading): 12
Tiny-BE rows kept: 4
        symbol formation_date  book_value        bm         op
906    SADBHIN     2017-06-30        71.0  0.021121  -2.633803
2416     MHRIL     2020-06-30        46.0  0.024658   6.304348
2566    INDIGO     2021-06-30       111.0  0.001767 -19.423423
2629  TATACOMM     2021-06-30       115.0  0.003797  33.400000


In [32]:
missing_td = formation["trade_days_12m"].isna()
print(missing_td.groupby(formation["formation_date"].dt.year).sum())

formation_date
2016    17
2017    14
2018    28
2019     9
2020    13
2021    22
2022    20
2023    11
2024    17
2025    21
2026    17
Name: trade_days_12m, dtype: int64


In [33]:
# ==========================================================
# STEP 4: Size Breakpoint and Eligibility
# ==========================================================

**Size breakpoint (FF):** one median of June ME per formation year, computed on
all stocks with a valid June ME (FF use the NYSE median). The same Small/Big label
is used in all three 2×3 sorts.

**Eligibility per sort (FF):**

| Sort | Required | Note |
|---|---|---|
| HML | BE > 0, valid B/M | Negative-BE firms excluded (FF) |
| RMW | BE > 0, valid OP | BE is OP's denominator; financials excluded (Step 3.6) |
| CMA | Valid asset growth | 12-month change, same statement basis (Step 2.13) |

No winsorizing of B/M, OP or Inv: sorts are rank-based, and FF don't trim.

In [34]:
# 4.1 Size: median June ME per formation year

size_bp = formation.groupby("formation_date")["me_june"].median()

formation["size_bp"] = formation["formation_date"].map(size_bp)

is_small = formation["me_june"] <= formation["size_bp"]
formation["SIZE"] = np.where(is_small, "S", "B")

print(formation.groupby(["formation_date", "SIZE"]).size().unstack())

SIZE              B    S
formation_date          
2016-06-30      249  249
2017-06-30      248  248
2018-06-30      249  250
2019-06-30      250  250
2020-06-30      249  250
2021-06-30      249  250
2022-06-30      249  250
2023-06-30      250  250
2024-06-30      250  250
2025-06-30      249  250
2026-06-30      249  250


In [35]:
# 4.2a Who can enter each 2x3 sort

pos_be = formation["book_value"] > 0

hml_ok = pos_be & (formation["bm"] > 0)
rmw_ok = pos_be & formation["op"].notna()
cma_ok = formation["inv"].notna()

eligible_hml = formation[hml_ok].copy()
eligible_rmw = formation[rmw_ok].copy()
eligible_cma = formation[cma_ok].copy()

print("HML eligible stock-years:", len(eligible_hml))
print("RMW eligible stock-years:", len(eligible_rmw))
print("CMA eligible stock-years:", len(eligible_cma))

HML eligible stock-years: 5271
RMW eligible stock-years: 4348
CMA eligible stock-years: 5223


In [36]:
#4.2b CMA needs two years of total assets -> first sort June 2016

CMA_START = "2016-06-30"
eligible_cma = eligible_cma[eligible_cma["formation_date"] >= CMA_START].copy()

print("CMA eligible stock-years:", len(eligible_cma))

CMA eligible stock-years: 5223


In [37]:
# 4.2c Use the latest June formation only if its fundamentals are complete
#      (fewer than 90% of the previous June's HML-eligible stocks = annual results not yet in the snapshot)

n_hml = eligible_hml.groupby("formation_date").size()
last, prev = n_hml.index[-1], n_hml.index[-2]

if n_hml[last] < 0.9 * n_hml[prev]:
    print("Latest formation incomplete, dropped:", last.date())
    for df in [eligible_hml, eligible_rmw, eligible_cma]:
        df.drop(df[df["formation_date"] == last].index, inplace=True)
else:
    print("Latest formation used:", last.date(), "|", n_hml[last], "vs", n_hml[prev], "stocks")

Latest formation used: 2026-06-30 | 496 vs 491 stocks


In [38]:
# 4.3 Eligible stocks per year (total and Big)

def count_by_year(df):
    total = df.groupby("formation_date").size()
    big = df[df["SIZE"] == "B"].groupby("formation_date").size()
    return pd.DataFrame({"total": total, "big": big})


counts = pd.concat(
    {
        "HML": count_by_year(eligible_hml),
        "RMW": count_by_year(eligible_rmw),
        "CMA": count_by_year(eligible_cma),
    },
    axis=1,
)

print(counts)

                 HML        RMW        CMA     
               total  big total  big total  big
formation_date                                 
2016-06-30       469  235   397  188   444  220
2017-06-30       468  237   401  189   462  233
2018-06-30       472  238   394  185   466  232
2019-06-30       471  238   391  184   469  237
2020-06-30       475  241   397  188   477  242
2021-06-30       474  241   389  192   473  239
2022-06-30       479  242   396  192   485  245
2023-06-30       486  244   395  186   481  241
2024-06-30       490  248   399  188   489  248
2025-06-30       491  247   397  184   487  244
2026-06-30       496  246   392  186   490  244


**Sample per factor**
- HML, RMW, CMA and SMB: from the June 2016 formation (first returns July 2016), the first June of the point-in-time universe
- The latest June formation is used only when its fundamentals are complete (Step 4.2c), so the sample extends automatically with each data refresh

In [39]:
# 4.4 Value ranges (check only, no trimming)

pct = [0.01, 0.30, 0.50, 0.70, 0.99]

print(eligible_hml["bm"].describe(percentiles=pct).round(3))
print(eligible_rmw["op"].describe(percentiles=pct).round(3))
print(eligible_cma["inv"].describe(percentiles=pct).round(3))

count    5271.000
mean        0.459
std         0.571
min         0.002
1%          0.029
30%         0.165
50%         0.270
70%         0.452
99%         2.897
max         8.670
Name: bm, dtype: float64
count    4348.000
mean        0.245
std         0.674
min       -19.423
1%         -0.295
30%         0.168
50%         0.228
70%         0.293
99%         1.016
max        33.400
Name: op, dtype: float64
count    5223.000
mean        0.164
std         0.406
min        -0.688
1%         -0.235
30%         0.061
50%         0.116
70%         0.181
99%         1.164
max        18.501
Name: inv, dtype: float64


In [40]:
# ==========================================================
# STEP 5: Market Factor (MKT)
# ==========================================================

$$
MKT_t = R_{m,t} - R_{f,t}, \qquad
R_{m,t} = \frac{\sum_i ME_{i,t-1} \, r_{i,t}}{\sum_i ME_{i,t-1}}
$$

| Item | Choice | FF convention |
|---|---|---|
| Universe | All point-in-time universe members with a return and lagged ME, **financials included** | All CRSP common stocks |
| Weights | ME at end of **previous** month (`me_lag`) | Same |
| Rf | FRED 3-month interbank rate (`Rf_monthly`), proxy for the 91-day T-bill | One-month T-bill |
| Returns | Total returns incl. dividends and corporate actions (Phase 1) | CRSP total returns |

MKT starts in July 2016, the first holding month of the point-in-time universe.

In [41]:
# 5.1 Value-weighted market return (weights = last month's ME)

mkt_data = final.dropna(subset=["ret", "me_lag"]).copy()
mkt_data = mkt_data[mkt_data["me_lag"] > 0]

# Each stock's return x its weight (before dividing by total ME)
mkt_data["w_ret"] = mkt_data["ret"] * mkt_data["me_lag"]

mkt = (
    mkt_data.groupby("date")
    .agg(
        w_ret=("w_ret", "sum"),
        total_me=("me_lag", "sum"),
        n_stocks=("symbol", "count"),
    )
    .reset_index()
)

mkt["market_return"] = mkt["w_ret"] / mkt["total_me"]

In [42]:
# 5.2 Risk-free rate and market excess return

# Rf must be a single value per month (same for all stocks)
rf_values = final.groupby("date")["Rf_monthly"].nunique()
print("Months with more than one Rf value:", (rf_values > 1).sum())  # must be 0

rf_by_date = final.groupby("date")["Rf_monthly"].first()

mkt["Rf"] = mkt["date"].map(rf_by_date)
mkt["MKT"] = mkt["market_return"] - mkt["Rf"]

print("Months without Rf:", mkt["Rf"].isna().sum())
print(mkt.loc[mkt["Rf"].isna(), "date"].dt.strftime("%Y-%m").tolist())

Months with more than one Rf value: 0
Months without Rf: 1
['2026-10']


Only edge months don't have Rf value (2011 and 2026) and that is fair.

In [43]:
# 5.3 Does MKT look like the Indian market?

# (a) Breadth: stocks per month, averaged by year
print(mkt.groupby(mkt["date"].dt.year)["n_stocks"].mean().round(0))


date
2016    497.0
2017    495.0
2018    496.0
2019    496.0
2020    495.0
2021    498.0
2022    498.0
2023    498.0
2024    499.0
2025    498.0
2026    497.0
Name: n_stocks, dtype: float64


In [44]:
# ----------------------------------------------------------
# 5.3(b) MKT premium (FF view) + wealth growth (investor view)
# ----------------------------------------------------------
w = mkt[mkt['date'] >= '2016-07-31'].dropna(subset=['Rf'])
n = len(w)
years = n / 12

# --- FF view: arithmetic mean of monthly MKT, with t-stat ---
mkt_mean = w['MKT'].mean()
mkt_sd = w['MKT'].std()
t_stat = mkt_mean / (mkt_sd / np.sqrt(n))

# --- Investor view: growth of Rs 100 ---
growth_m = (1 + w['market_return']).prod()
growth_f = (1 + w['Rf']).prod()

cagr_m = growth_m ** (1 / years) - 1
cagr_f = growth_f ** (1 / years) - 1
geo_excess = (1 + cagr_m) / (1 + cagr_f) - 1

# --- Consistency check: arithmetic ~ CAGR + variance / 2 ---
arith_m = w['market_return'].mean() * 12
drag = (w['market_return'].std() * np.sqrt(12)) ** 2 / 2

print(w['date'].min().date(), '->', w['date'].max().date(), '|', n, 'months')
print()
print('FF VIEW (used in the model)')
print(f'  MKT mean monthly     : {mkt_mean * 100:.2f}%')
print(f'  MKT mean annualised  : {mkt_mean * 12 * 100:.2f}%')
print(f'  t-stat               : {t_stat:.2f}')
print()
print('INVESTOR VIEW (sanity check)')
print(f'  Rs 100 in market     -> Rs {100 * growth_m:.0f}  (CAGR {cagr_m * 100:.2f}%)')
print(f'  Rs 100 in T-bills    -> Rs {100 * growth_f:.0f}  (CAGR {cagr_f * 100:.2f}%)')
print(f'  Geometric excess     : {geo_excess * 100:.2f}% per year')
print()
print('CHECK')
print(f'  Arithmetic market    : {arith_m * 100:.2f}%')
print(f'  CAGR + drag          : {(cagr_m + drag) * 100:.2f}%   (should be close)')

2016-07-31 -> 2026-09-30 | 123 months

FF VIEW (used in the model)
  MKT mean monthly     : 0.70%
  MKT mean annualised  : 8.41%
  t-stat               : 1.58

INVESTOR VIEW (sanity check)
  Rs 100 in market     -> Rs 359  (CAGR 13.29%)
  Rs 100 in T-bills    -> Rs 177  (CAGR 5.74%)
  Geometric excess     : 7.13% per year

CHECK
  Arithmetic market    : 14.01%
  CAGR + drag          : 14.73%   (should be close)


In [45]:
# (c) Best and worst months -> should match known market events
show = ["date", "market_return", "n_stocks"]
print(mkt.nsmallest(5, "market_return")[show].round(3))
print(mkt.nlargest(5, "market_return")[show].round(3))

          date  market_return  n_stocks
44  2020-03-31         -0.227       496
116 2026-03-31         -0.112       497
103 2025-02-28         -0.091       499
26  2018-09-30         -0.089       497
99  2024-10-31         -0.070       500
          date  market_return  n_stocks
45  2020-04-30          0.137       490
117 2026-04-30          0.117       497
52  2020-11-30          0.107       497
72  2022-07-31          0.098       499
47  2020-06-30          0.090       489


In [46]:
# 5.3(d) Which stock-months drag the market return most?

month_total = mkt_data.groupby('date')['me_lag'].transform('sum')
mkt_data['weight'] = mkt_data['me_lag'] / month_total
mkt_data['contrib'] = mkt_data['weight'] * mkt_data['ret']

show = ['date', 'symbol', 'ret', 'weight', 'contrib']
print(mkt_data.nsmallest(15, 'contrib')[show].round(4))

            date      symbol     ret  weight  contrib
24150 2020-03-31    HDFCBANK -0.2681  0.0470  -0.0126
47375 2020-03-31    RELIANCE -0.1617  0.0614  -0.0099
6280  2020-03-31  BAJFINANCE -0.5027  0.0196  -0.0098
47358 2018-10-31    RELIANCE -0.1564  0.0559  -0.0087
47382 2020-10-31    RELIANCE -0.0805  0.1040  -0.0084
26026 2020-03-31   ICICIBANK -0.3489  0.0234  -0.0082
1428  2023-02-28    ADANIENT -0.5414  0.0139  -0.0075
47405 2022-09-30    RELIANCE -0.0986  0.0703  -0.0069
55034 2018-10-31         TCS -0.1105  0.0626  -0.0069
49208 2020-03-31        SBIN -0.3503  0.0197  -0.0069
23977 2020-03-31        HDFC -0.2494  0.0274  -0.0068
4998  2023-01-31        ATGL -0.4291  0.0159  -0.0068
5576  2020-03-31    AXISBANK -0.4565  0.0143  -0.0065
1507  2022-05-31  ADANIGREEN -0.3448  0.0187  -0.0064
4999  2023-02-28        ATGL -0.6781  0.0095  -0.0064


In [47]:
# 5.4 Is any single stock dominating the market return?

top_idx = mkt_data.groupby("date")["weight"].idxmax()
top = mkt_data.loc[top_idx, ["date", "symbol", "weight"]]

print(top["weight"].describe().round(3))
print(top.nlargest(5, "weight").round(3))

count    124.000
mean       0.058
std        0.014
min        0.039
25%        0.047
50%        0.056
75%        0.067
max        0.104
Name: weight, dtype: float64
            date    symbol  weight
47382 2020-10-31  RELIANCE   0.104
47380 2020-08-31  RELIANCE   0.100
47381 2020-09-30  RELIANCE   0.097
47383 2020-11-30  RELIANCE   0.094
47379 2020-07-31  RELIANCE   0.088


Highest weight of less than 20% is fine from the Indian context.

In [48]:
# 5.5 Clean market-factor table for later steps

mkt_factor = mkt[['date', 'market_return', 'Rf', 'MKT']].copy()

print(mkt_factor[60:65])
print(mkt_factor.isnull().sum())

mkt_factor.to_csv(PROCESSED_DIR / 'mkt_factor.csv', index=False)

         date  market_return        Rf       MKT
60 2021-07-31       0.013389  0.002865  0.010524
61 2021-08-31       0.066473  0.002855  0.063619
62 2021-09-30       0.039201  0.002791  0.036411
63 2021-10-31      -0.003528  0.002755 -0.006283
64 2021-11-30      -0.021589  0.002888 -0.024478
date             0
market_return    0
Rf               1
MKT              1
dtype: int64


### Step 5 Summary: Market Factor (MKT)

**Construction:** MKTₜ = R_m,t − R_f,t, built monthly with simple returns. R_m is the
value-weighted return of all panel stocks (financials included), weighted by last
month's ME (`me_lag`). Rf is the monthly 91-day T-bill rate. Returns are total returns from Phase 1.

**Coverage:** Dec 2011 to Jul 2026 (176 months); Rf missing only at the 2011 and 2026 edges.

**Results (Jul 2016 – Jul 2026, 121 months)**

| Measure | Value |
|---|---|
| MKT mean, annualised (t-stat) | 1.00 |
| ₹100 in market → ₹262 | CAGR 10.03% |
| ₹100 in T-bills → ₹176 | CAGR 5.75% |

**Checks:** worst month __March '20__ (-21.9%), best month __January '12__ (15.9%); largest single-stock
weight 16.8% of SBIN (< 20%).

**Limitations:** full-cap rather than free-float weights; Rf proxy (FRED interbank rate).

**Output:** `mkt_factor.csv` (`date`, `market_return`, `Rf`, `MKT`), used in Step 11 and Phase 3.

In [49]:
# ==========================================================
# STEP 6: Helper Function for 2×3 Sorts
# ==========================================================

This function constructs:
- **HML** portfolios
- **RMW** portfolios
- **CMA** portfolios

using:

- Small / Big
- Low / Neutral / High

In [50]:
# methodology: NSE-adapted FF5 using median size and 30/70 characteristic breakpoints

In [51]:
# Generic 2x3 Sort Function

def assign_2x3_portfolios(df, variable, low_label='L', high_label='H'):

    df = df.copy()
    df = df.dropna(subset=[variable, 'me_june'])

    # valid positive market capitalization
    df = df[df['me_june'] > 0].copy()

    # CHARACTERISTIC BREAKPOINTS
    # ------------------------------------------------------
    # 30th and 70th percentiles ONLY among BIG stocks within each formation year.
    # These breakpoints are then applied to BOTH Small and Big stocks.
    # ------------------------------------------------------
    big = df[df['SIZE'] == 'B'].copy()

    low_breakpoint = (
        big.groupby('formation_date')[variable]
        .quantile(0.30)
        .rename('low_breakpoint')
    )

    high_breakpoint = (
        big.groupby('formation_date')[variable]
        .quantile(0.70)
        .rename('high_breakpoint')
    )

    # Attach Big-stock breakpoints to ALL stocks
    df = df.merge(
        pd.concat([low_breakpoint, high_breakpoint], axis=1),
        on='formation_date',
        how='left'
    )

    # CHARACTERISTIC BUCKETS
    df['CHAR'] = 'N'
    df.loc[df[variable] <= df['low_breakpoint'], 'CHAR'] = low_label
    df.loc[df[variable] >= df['high_breakpoint'], 'CHAR'] = high_label

    # FINAL 2x3 PORTFOLIO
    df['PORT'] = df['SIZE'] + df['CHAR']

    return df

Assigns each stock to one of six portfolios: **Size** (S/B) × **Characteristic** (Low/Neutral/High).

- Size: from Step 4.1 (one median June ME per year, shared by all sorts)
- Characteristic breakpoints: 30th / 70th percentiles of **Big stocks only**,
  applied to all stocks (FF international convention, the analog of NYSE breakpoints)
- Used for HML (B/M), RMW (OP) and CMA (Inv)

In [52]:
# ==========================================================
# STEP 7: HML Construction
# ==========================================================

$$
HML = \tfrac{1}{2}(SH + BH) - \tfrac{1}{2}(SL + BL)
$$

- **H** = Value (high B/M), **L** = Growth (low B/M)
- Eligible stocks: Step 4.2 (`eligible_hml`: BE > 0, valid B/M)
- Portfolios formed end of June t, held July t → June t+1
- Monthly returns value-weighted with **last month's ME** (`me_lag`)
- A portfolio-month with fewer than `MIN_STOCKS` stocks is set to NaN

In [53]:
# 7.1 Assign each eligible stock to a 2x3 portfolio

# eligible_hml comes from Step 4.2 -> not redefined here
hml_assign = assign_2x3_portfolios(
    eligible_hml,
    'bm',
    low_label='L',
    high_label='H'
)

print(hml_assign['PORT'].value_counts().sort_index())

PORT
BH     801
BL     801
BN    1055
SH    1006
SL     443
SN    1165
Name: count, dtype: int64


In [54]:
# 7.2 Each stock keeps its portfolio from July t to June t+1

keys = ['symbol', 'formation_date']

hml_monthly = final.merge(
    hml_assign[keys + ['PORT']],
    on=keys,
    how='inner'          # keep only stocks that have a portfolio
)

print(hml_monthly['PORT'].value_counts().sort_index())

PORT
BH     9006
BL     9020
BN    11876
SH    11239
SL     4931
SN    13150
Name: count, dtype: int64


In [55]:
# 7.3 Monthly VW returns (weights = last month's ME)

def vw_portfolio_returns(monthly):
    df = monthly.dropna(subset=['ret', 'me_lag'])
    df = df[df['me_lag'] > 0].copy()

    df['w_ret'] = df['ret'] * df['me_lag']

    ports = df.groupby(['date', 'PORT']).agg(
        w_ret=('w_ret', 'sum'),
        total_me=('me_lag', 'sum'),
        n_stocks=('symbol', 'count'),
    ).reset_index()

    ports['ret'] = ports['w_ret'] / ports['total_me']
    ports.loc[ports['n_stocks'] < MIN_STOCKS, 'ret'] = np.nan
    return ports

hml_ports = vw_portfolio_returns(hml_monthly)
print(hml_ports.head())

        date PORT          w_ret      total_me  n_stocks       ret
0 2016-07-31   BH  186786.761967  2.436025e+06        71  0.076677
1 2016-07-31   BL  115225.040166  2.549674e+06        71  0.045192
2 2016-07-31   BN  164335.989404  3.513698e+06        93  0.046770
3 2016-07-31   SH    9278.074869  2.572316e+05        87  0.036069
4 2016-07-31   SL    1992.127171  8.650658e+04        26  0.023029


In [56]:
# 7.4 Six portfolios side by side -> HML

hml_pivot = hml_ports.pivot(index='date', columns='PORT', values='ret')

# All six columns must exist (SN, BN needed later for SMB)
hml_pivot = hml_pivot.reindex(columns=['SL', 'SN', 'SH', 'BL', 'BN', 'BH'])

value = (hml_pivot['SH'] + hml_pivot['BH']) / 2
growth = (hml_pivot['SL'] + hml_pivot['BL']) / 2
hml_pivot['HML'] = value - growth

print(hml_pivot.head())

hml_factor = hml_pivot[['HML']].reset_index()

print(hml_factor.head())

PORT              SL        SN        SH        BL        BN        BH  \
date                                                                     
2016-07-31  0.023029  0.045950  0.036069  0.045192  0.046770  0.076677   
2016-08-31  0.026762  0.026598  0.021585  0.004043  0.011005  0.048387   
2016-09-30 -0.000355  0.000322  0.003097 -0.016707 -0.010720 -0.004701   
2016-10-31  0.034913  0.049823  0.054691 -0.007917  0.017079  0.039669   
2016-11-30 -0.069829 -0.076507 -0.071962 -0.063435 -0.045704 -0.023852   

PORT             HML  
date                  
2016-07-31  0.022263  
2016-08-31  0.019584  
2016-09-30  0.007729  
2016-10-31  0.033682  
2016-11-30  0.018725  
PORT       date       HML
0    2016-07-31  0.022263
1    2016-08-31  0.019584
2    2016-09-30  0.007729
3    2016-10-31  0.033682
4    2016-11-30  0.018725


In [57]:
# 7.5 Are portfolios well populated? Does HML look sensible?

# (a) Stocks per portfolio per formation year
per_year = hml_assign.groupby(['formation_date', 'PORT']).size()
print(per_year.unstack(fill_value=0))

# (b) Months where HML is missing
print('HML months:', hml_factor['HML'].notna().sum())
print('HML missing:', hml_factor['HML'].isna().sum())

# (c) Premium: mean, t-stat
h = hml_factor['HML'].dropna()
t_stat = h.mean() / (h.std() / np.sqrt(len(h)))
print(f'HML mean annualised: {h.mean() * 12 * 100:.2f}%  |  t-stat: {t_stat:.2f}')

PORT            BH  BL  BN   SH  SL   SN
formation_date                          
2016-06-30      71  71  93   87  26  121
2017-06-30      71  71  95   89  40  102
2018-06-30      72  72  94   79  50  105
2019-06-30      72  72  94   88  36  109
2020-06-30      73  73  95  119  21   94
2021-06-30      73  73  95  101  32  100
2022-06-30      73  73  96   90  44  103
2023-06-30      73  73  98   91  46  105
2024-06-30      75  75  98   71  56  115
2025-06-30      74  74  99   90  45  109
2026-06-30      74  74  98  101  47  102
HML months: 124
HML missing: 0
HML mean annualised: 6.06%  |  t-stat: 1.68


In [58]:
# 7.6 How many company-months came under which PORT how many times?

hml_monthly.head()

,symbol,date,ret,me,me_lag,Rf_monthly,is_financial,formation_date,PORT
0,360ONE,2020-07-31,0.047468,9308.677465,8886.840644,0.002759,True,2020-06-30,BN
1,360ONE,2020-08-31,-0.009007,8902.061560,9308.677465,0.002681,True,2020-06-30,BN
2,360ONE,2020-09-30,-0.060332,8364.980660,8902.061560,0.002695,True,2020-06-30,BN
3,360ONE,2020-10-31,-0.043462,8001.418205,8364.980660,0.002729,True,2020-06-30,BN
4,360ONE,2020-11-30,0.104027,8833.784879,8001.418205,0.002711,True,2020-06-30,BN


In [59]:
port_symbols = (hml_monthly
                .groupby(['PORT', 'symbol'])
                .size()
                .reset_index(name='months'))

port_symbols = port_symbols.sort_values(['PORT', 'months'],
                                        ascending=[True, False])

In [60]:
with pd.option_context('display.max_rows', None):
    display(port_symbols)

,PORT,symbol,months
16,BH,BAJAJHLDNG,124
18,BH,BANKBARODA,124
19,BH,BANKINDIA,124
26,BH,CANBK,124
46,BH,FEDERALBNK,124
49,BH,GAIL,124
57,BH,GRASIM,124
62,BH,HINDALCO,124
66,BH,IDBI,124
68,BH,IDFCFIRSTB,124


In [61]:
# ==========================================================
# STEP 8: RMW Construction
# ==========================================================

$$
RMW = \tfrac{1}{2}(SR + BR) - \tfrac{1}{2}(SW + BW)
$$

- **R** = Robust (high OP), **W** = Weak (low OP)
- OP = (Operating profit − Interest) / BE, FY ending ≤ March t
- Eligible stocks: Step 4.2 (`eligible_rmw`: BE > 0, valid OP); **financials excluded** (Step 3.6)
- Portfolios formed end of June t, held July t → June t+1
- Monthly returns value-weighted with `me_lag`; portfolio-months below `MIN_STOCKS` set to NaN

In [62]:
# 8.1 Assign each eligible stock to a 2x3 portfolio

# eligible_rmw comes from Step 4.2 -> not redefined here
rmw_assign = assign_2x3_portfolios(
    eligible_rmw,
    'op',
    low_label='W',
    high_label='R'
)

print(rmw_assign['PORT'].value_counts().sort_index())

# Financials must not be in RMW (Step 3.6)
fin_in_rmw = rmw_assign['is_financial'].fillna(False).astype(bool).sum()
print('Financial stock-years in RMW:', fin_in_rmw)       # must be 0

PORT
BN    818
BR    622
BW    622
SN    888
SR    482
SW    916
Name: count, dtype: int64
Financial stock-years in RMW: 0


In [63]:
# 8.2 Each stock keeps its portfolio from July t to June t+1

keys = ['symbol', 'formation_date']

rmw_monthly = final.merge(
    rmw_assign[keys + ['PORT']],
    on=keys,
    how='inner'
)

print(rmw_monthly['PORT'].value_counts().sort_index())

PORT
BN     9224
BR     7016
BW     7002
SN    10059
SR     5485
SW    10198
Name: count, dtype: int64


In [64]:
# 8.3 Monthly VW returns (same function as Step 7.3)

rmw_ports = vw_portfolio_returns(rmw_monthly)
print(rmw_ports.head())

        date PORT          w_ret      total_me  n_stocks       ret
0 2016-07-31   BN  117722.119424  2.519218e+06        74  0.046730
1 2016-07-31   BR  118763.231673  2.732395e+06        57  0.043465
2 2016-07-31   BW  100532.922903  1.486331e+06        57  0.067638
3 2016-07-31   SN    8626.789032  2.331074e+05        80  0.037008
4 2016-07-31   SR    5813.514511  1.400703e+05        47  0.041504


In [65]:
# 8.4 Six portfolios side by side -> RMW

rmw_pivot = rmw_ports.pivot(index='date', columns='PORT', values='ret')

# All six columns must exist (SN, BN needed later for SMB)
rmw_pivot = rmw_pivot.reindex(columns=['SW', 'SN', 'SR', 'BW', 'BN', 'BR'])

robust = (rmw_pivot['SR'] + rmw_pivot['BR']) / 2
weak = (rmw_pivot['SW'] + rmw_pivot['BW']) / 2
rmw_pivot['RMW'] = robust - weak

rmw_factor = rmw_pivot[['RMW']].reset_index()
print(rmw_factor.tail())

PORT       date       RMW
119  2026-06-30 -0.015032
120  2026-07-31  0.027861
121  2026-08-31 -0.005834
122  2026-09-30 -0.022322
123  2026-10-31  0.041862


In [66]:
# 8.5 Are portfolios well populated? Does RMW look sensible?

# (a) Stocks per portfolio per formation year
per_year = rmw_assign.groupby(['formation_date', 'PORT']).size()
print(per_year.unstack(fill_value=0))

# (b) Months where RMW is missing
print('RMW months:', rmw_factor['RMW'].notna().sum())
print('RMW missing:', rmw_factor['RMW'].isna().sum())

# (c) Premium: mean, t-stat
r = rmw_factor['RMW'].dropna()
t_stat = r.mean() / (r.std() / np.sqrt(len(r)))
print(f'RMW mean annualised: {r.mean() * 12 * 100:.2f}%  |  t-stat: {t_stat:.2f}')

PORT            BN  BR  BW  SN  SR  SW
formation_date                        
2016-06-30      74  57  57  80  47  82
2017-06-30      75  57  57  80  40  92
2018-06-30      73  56  56  78  46  85
2019-06-30      74  55  55  82  51  74
2020-06-30      74  57  57  81  51  77
2021-06-30      76  58  58  79  40  78
2022-06-30      76  58  58  83  43  78
2023-06-30      74  56  56  89  48  72
2024-06-30      74  57  57  75  44  92
2025-06-30      74  55  55  89  35  89
2026-06-30      74  56  56  72  37  97
RMW months: 124
RMW missing: 0
RMW mean annualised: -1.33%  |  t-stat: -0.48


In [67]:
rmw_symbols = (rmw_monthly
               .groupby(['PORT', 'symbol'])
               .size()
               .reset_index(name='months')
               .sort_values(['PORT', 'months'], ascending=[True, False]))

print(rmw_symbols.to_string())

     PORT      symbol  months
31     BN  BALKRISIND     124
87     BN     HAVELLS     124
46     BN       CIPLA     120
209    BN       WIPRO     120
14     BN       ALKEM     112
25     BN      ASTRAL     112
54     BN       DABUR     112
146    BN        ONGC     112
166    BN  SCHAEFFLER     112
185    BN       TECHM     112
97     BN         IGL     108
123    BN          LT     100
127    BN         M&M     100
134    BN     MPHASIS     100
192    BN  TORNTPOWER     100
9      BN  ADANIPORTS      96
60     BN     DRREDDY      96
199    BN         UPL      96
58     BN    DIVISLAB      88
59     BN       DMART      88
79     BN    GODREJCP      88
142    BN        NMDC      88
167    BN    SHREECEM      88
212    BN   ZYDUSLIFE      88
154    BN       PIIND      84
180    BN     SYNGENE      84
81     BN      GRASIM      76
129    BN      MARUTI      76
143    BN        NTPC      76
173    BN         SRF      76
176    BN   SUNPHARMA      76
196    BN  ULTRACEMCO      76
3      BN 

In [68]:
# ==========================================================
# STEP 9: CMA FACTOR (FULLY SELF-CONTAINED)
# ==========================================================

$$
CMA = \tfrac{1}{2}(SC + BC) - \tfrac{1}{2}(SA + BA)
$$

- **C** = Conservative (low asset growth), **A** = Aggressive (high asset growth)
- Inv = (TA_Mar t − TA_Mar t−1) / TA_Mar t−1; 12-month change on the same statement basis (Step 2.13)
- Eligible stocks: Step 4.2 (`eligible_cma`); first formation **June 2016** (Step 4.2b),
  since asset growth needs two years of total assets
- Financials included (asset growth is comparable across firms)
- Portfolios formed end of June t, held July t → June t+1
- Monthly returns value-weighted with `me_lag`; portfolio-months below `MIN_STOCKS` set to NaN

In [69]:
# 9.1 Assign each eligible stock to a 2x3 portfolio

# eligible_cma comes from Step 4.2 / 4.2b -> not redefined here
cma_assign = assign_2x3_portfolios(
    eligible_cma,
    'inv',
    low_label='C',       # low asset growth  = Conservative
    high_label='A'       # high asset growth = Aggressive
)

print(cma_assign['PORT'].value_counts().sort_index())
print('First formation:', cma_assign['formation_date'].min().date())   # expect 2016-06-30

PORT
BA     790
BC     790
BN    1045
SA     812
SC     949
SN     837
Name: count, dtype: int64
First formation: 2016-06-30


In [70]:
# 9.2 Each stock keeps its portfolio from July t to June t+1

keys = ['symbol', 'formation_date']

cma_monthly = final.merge(
    cma_assign[keys + ['PORT']],
    on=keys,
    how='inner'
)

print(cma_monthly['PORT'].value_counts().sort_index())

PORT
BA     8896
BC     8885
BN    11753
SA     9094
SC    10707
SN     9359
Name: count, dtype: int64


In [71]:
# 9.3 Monthly VW returns (same function as Step 7.3)

cma_ports = vw_portfolio_returns(cma_monthly)
print(cma_ports.head())

        date PORT          w_ret      total_me  n_stocks       ret
0 2016-07-31   BA  155158.895217  2.825970e+06        66  0.054905
1 2016-07-31   BC  133549.932259  2.073203e+06        66  0.064417
2 2016-07-31   BN  141740.818602  3.157896e+06        88  0.044885
3 2016-07-31   SA    9581.769511  1.644271e+05        57  0.058274
4 2016-07-31   SC   10052.252005  2.756937e+05        90  0.036462


In [72]:
# 9.4 Six portfolios side by side -> CMA

cma_pivot = cma_ports.pivot(index='date', columns='PORT', values='ret')

# All six columns must exist (SN, BN needed later for SMB)
cma_pivot = cma_pivot.reindex(columns=['SC', 'SN', 'SA', 'BC', 'BN', 'BA'])

conservative = (cma_pivot['SC'] + cma_pivot['BC']) / 2
aggressive = (cma_pivot['SA'] + cma_pivot['BA']) / 2
cma_pivot['CMA'] = conservative - aggressive

cma_factor = cma_pivot[['CMA']].reset_index()
print(cma_factor.head())

PORT       date       CMA
0    2016-07-31 -0.006150
1    2016-08-31  0.018471
2    2016-09-30  0.000984
3    2016-10-31  0.008754
4    2016-11-30  0.037563


In [73]:
# 9.5 Are portfolios well populated? Does CMA look sensible?

# (a) Stocks per portfolio per formation year
per_year = cma_assign.groupby(['formation_date', 'PORT']).size()
print(per_year.unstack(fill_value=0))

# (b) Months where CMA is missing
print('CMA months:', cma_factor['CMA'].notna().sum())
print('CMA missing:', cma_factor['CMA'].isna().sum())

# (c) Premium: mean, t-stat
c = cma_factor['CMA'].dropna()
t_stat = c.mean() / (c.std() / np.sqrt(len(c)))
print(f'CMA mean annualised: {c.mean() * 12 * 100:.2f}%  |  t-stat: {t_stat:.2f}')


PORT            BA  BC  BN  SA   SC  SN
formation_date                         
2016-06-30      66  66  88  57   90  77
2017-06-30      70  70  93  55   97  77
2018-06-30      70  70  92  55   85  94
2019-06-30      71  71  95  67   86  79
2020-06-30      73  73  96  73  104  58
2021-06-30      72  72  95  64   94  76
2022-06-30      74  74  97  95   66  79
2023-06-30      73  73  95  92   80  68
2024-06-30      75  75  98  89   81  71
2025-06-30      73  73  98  87   82  74
2026-06-30      73  73  98  78   84  84
CMA months: 124
CMA missing: 0
CMA mean annualised: 1.34%  |  t-stat: 0.59


In [74]:
# (d) Asset growth distribution per year (old describe cell)
pct = [0.30, 0.50, 0.70]
print(cma_assign.groupby('formation_date')['inv'].describe(percentiles=pct).round(3))

# (e) CMA vs HML correlation (FF 2015: strongly positive in the US)
both = hml_factor.merge(cma_factor, on='date').dropna()
print('Corr(HML, CMA):', round(both['HML'].corr(both['CMA']), 2))

                count   mean    std    min    30%    50%    70%     max
formation_date                                                         
2016-06-30      444.0  0.121  0.221 -0.688  0.026  0.089  0.156   1.994
2017-06-30      462.0  0.171  0.877 -0.377  0.048  0.097  0.165  18.501
2018-06-30      466.0  0.173  0.377 -0.464  0.067  0.120  0.179   5.767
2019-06-30      469.0  0.145  0.319 -0.670  0.052  0.102  0.176   5.585
2020-06-30      477.0  0.114  0.191 -0.393  0.032  0.085  0.148   1.544
2021-06-30      473.0  0.150  0.315 -0.551  0.032  0.099  0.162   3.492
2022-06-30      485.0  0.207  0.497 -0.511  0.080  0.136  0.214   9.682
2023-06-30      481.0  0.157  0.225 -0.323  0.064  0.116  0.184   2.513
2024-06-30      489.0  0.174  0.212 -0.380  0.081  0.132  0.199   1.853
2025-06-30      487.0  0.199  0.474 -0.633  0.084  0.132  0.189   9.337
2026-06-30      490.0  0.188  0.247 -0.417  0.101  0.142  0.203   3.674
Corr(HML, CMA): 0.3


In [75]:
cma_symbols = (cma_monthly
               .groupby(['PORT', 'symbol'])
               .size()
               .reset_index(name='months')
               .sort_values(['PORT', 'months'], ascending=[True, False]))

print(cma_symbols.to_string())

     PORT      symbol  months
43     BA  BAJFINANCE     108
40     BA  BAJAJFINSV      96
34     BA      AUBANK      88
294    BA       TITAN      88
155    BA      INDIGO      84
300    BA    TVSMOTOR      84
189    BA         LTM      76
246    BA    PRESTIGE      76
95     BA   EICHERMOT      72
150    BA         IGL      72
190    BA        LTTS      72
214    BA      NAUKRI      72
238    BA       PIIND      72
10     BA  ADANIGREEN      64
26     BA    ASHOKLEY      64
68     BA    CHOLAFIN      64
122    BA  GODREJPROP      64
210    BA  MUTHOOTFIN      64
231    BA  PERSISTENT      64
6      BA       ABFRL      60
45     BA  BANDHANBNK      60
55     BA      BIOCON      60
59     BA   BRITANNIA      60
134    BA    HDFCBANK      60
182    BA  LALPATHLAB      60
204    BA        MFSL      60
269    BA         SRF      60
304    BA    UNOMINDA      60
8      BA  ADANIENSOL      52
11     BA  ADANIPORTS      52
31     BA        ATGL      52
91     BA       DMART      52
169    BA 

2095   SN         UBL       4


In [76]:
# ==========================================================
# STEP 10: SMB Construction (Canonical FF5)
# ==========================================================

$$
SMB = \tfrac{1}{3}\left(SMB_{B/M} + SMB_{OP} + SMB_{Inv}\right)
$$

Each component = average of the 3 Small portfolios − average of the 3 Big portfolios
from that 2×3 sort (canonical FF5).

- SMB is NaN if any component is missing; with the point-in-time universe all three components start in July 2016
- Size breakpoint: one June median for all sorts (Step 4.1)

In [77]:
# 10.1 Size spread from each 2x3 sort

def size_spread(pivot, small_cols, big_cols):
    small = pivot[small_cols].mean(axis=1, skipna=False) # skipna=False: if one portfolio is missing, the spread is NaN
    big = pivot[big_cols].mean(axis=1, skipna=False)
    return small - big

smb_hml = size_spread(hml_pivot, ['SL', 'SN', 'SH'], ['BL', 'BN', 'BH'])
smb_rmw = size_spread(rmw_pivot, ['SW', 'SN', 'SR'], ['BW', 'BN', 'BR'])
smb_cma = size_spread(cma_pivot, ['SC', 'SN', 'SA'], ['BC', 'BN', 'BA'])

In [78]:
# 10.2 SMB = average of the three size spreads

smb = pd.concat([smb_hml, smb_rmw, smb_cma], axis=1)
smb.columns = ['SMB_HML', 'SMB_RMW', 'SMB_CMA']

smb['SMB'] = smb.mean(axis=1, skipna=False)

smb_factor = smb[['SMB']].reset_index()
print(smb_factor.dropna().head())

        date       SMB
0 2016-07-31 -0.017881
1 2016-08-31  0.008015
2 2016-09-30  0.012967
3 2016-10-31  0.028923
4 2016-11-30 -0.031572


In [79]:
# 10.3 Coverage, premium, consistency

# (a) Which component is missing, and when?
print(smb[['SMB_HML', 'SMB_RMW', 'SMB_CMA']].isna().sum())
print('SMB first month:', smb['SMB'].first_valid_index().date())    # expect 2016-07-31

# (b) Premium: mean, t-stat
s = smb['SMB'].dropna()
t_stat = s.mean() / (s.std() / np.sqrt(len(s)))
print(f'SMB mean annualised: {s.mean() * 12 * 100:.2f}%  |  t-stat: {t_stat:.2f}')

# (c) The three spreads should move together
print(smb[['SMB_HML', 'SMB_RMW', 'SMB_CMA']].corr().round(2))

SMB_HML    0
SMB_RMW    0
SMB_CMA    0
dtype: int64
SMB first month: 2016-07-31
SMB mean annualised: 2.26%  |  t-stat: 0.64
         SMB_HML  SMB_RMW  SMB_CMA
SMB_HML     1.00     0.97     0.99
SMB_RMW     0.97     1.00     0.96
SMB_CMA     0.99     0.96     1.00


In [80]:
# All three size spreads start in July 2016: check (a) should show 0 missing for each

In [81]:
# 10.4 Missing months per portfolio (lists renamed: no clash)

hml_cols = ['SL', 'SN', 'SH', 'BL', 'BN', 'BH']
rmw_cols = ['SW', 'SN', 'SR', 'BW', 'BN', 'BR']
cma_cols = ['SC', 'SN', 'SA', 'BC', 'BN', 'BA']

for name, pivot, cols in [('HML', hml_pivot, hml_cols),
                          ('RMW', rmw_pivot, rmw_cols),
                          ('CMA', cma_pivot, cma_cols)]:
    print(f'\n{name} portfolios, missing months:')
    print(pivot[cols].isna().sum().to_string())


HML portfolios, missing months:
PORT
SL    0
SN    0
SH    0
BL    0
BN    0
BH    0

RMW portfolios, missing months:
PORT
SW    0
SN    0
SR    0
BW    0
BN    0
BR    0

CMA portfolios, missing months:
PORT
SC    0
SN    0
SA    0
BC    0
BN    0
BA    0


In [82]:
# ==========================================================
# STEP 11: Final FF5 Factor Dataset
# ==========================================================

One row per month: `MKT, SMB, HML, RMW, CMA, Rf` (decimals, e.g. 0.012 = 1.2%).
Common sample = months where **all five factors** exist (from July 2016).
Saved to `data/processed/ff5_factors.csv` for Phase 3.

In [83]:
# 11.1 Put all factors side by side, aligned on date

ff5 = pd.concat([
    mkt_factor.set_index('date')[['MKT', 'Rf']],
    smb_factor.set_index('date')['SMB'],
    hml_factor.set_index('date')['HML'],
    rmw_factor.set_index('date')['RMW'],
    cma_factor.set_index('date')['CMA'],
], axis=1)

ff5 = ff5[['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'Rf']]   # FF column order

print(len(ff5))
print(ff5.isnull().sum())

124
MKT    1
SMB    0
HML    0
RMW    0
CMA    0
Rf     1
dtype: int64


In [84]:
for col in ff5.columns:
    s = ff5[col]
    first, last = s.first_valid_index(), s.last_valid_index()
    gaps = s.loc[first:last].isna().sum()
    print(f'{col:4}  {first.date()} -> {last.date()}  | NaN inside: {gaps}')

MKT   2016-07-31 -> 2026-09-30  | NaN inside: 0
SMB   2016-07-31 -> 2026-10-31  | NaN inside: 0
HML   2016-07-31 -> 2026-10-31  | NaN inside: 0
RMW   2016-07-31 -> 2026-10-31  | NaN inside: 0
CMA   2016-07-31 -> 2026-10-31  | NaN inside: 0
Rf    2016-07-31 -> 2026-09-30  | NaN inside: 0


NaN values are only at the edges. Checked!

In [85]:
ff5[80:90]

,MKT,SMB,HML,RMW,CMA,Rf
date,,,,,,
2023-03-31,-0.000242,-0.032100,0.014064,-0.018941,-0.005031,0.005578
2023-04-30,0.044249,0.035096,0.041215,-0.019437,-0.014291,0.005729
2023-05-31,0.035525,0.000967,-0.028573,-0.006575,-0.000059,0.005677
2023-06-30,0.037020,0.027080,0.012356,-0.003389,-0.000112,0.005719
2023-07-31,0.037627,0.030556,0.064879,-0.043359,0.012052,0.005637
2023-08-31,-0.006796,0.064028,0.006673,-0.008840,-0.005029,0.005603
2023-09-30,0.024924,-0.012493,0.054909,0.010978,0.023261,0.005667
2023-10-31,-0.034166,0.012956,-0.000429,0.003789,0.006253,0.005698
2023-11-30,0.068424,0.004119,0.013920,-0.029538,0.005091,0.005745


***What one SMB value means?***

SMB in month t = how much the small-stock side returned minus the big-stock side, in that month.
S = (SL+SN+SH)/3 in that duration

Similarly in HML, RMW, and CMA.

***Other values***

- Rf is the monthly risk-free return (FRED 3-month interbank rate, proxy for the 91-day T-bill).

- MKT = Rm - Rf, where Rm is the value-weighted return of all universe members (financials included), weighted by last month’s ME (me_lag)

In [86]:
# 11.2 Keep months where ALL factors exist; check no gaps

ff5 = ff5.dropna().sort_index()
ff5.index.name = 'date'

print(ff5.index.min().date(), '->', ff5.index.max().date(), '|', len(ff5), 'months')

# Every step should be exactly one month
month_id = ff5.index.year * 12 + ff5.index.month
gaps = pd.Series(month_id).diff().value_counts()

print(gaps)                                   # only 1.0 expected

2016-07-31 -> 2026-09-30 | 123 months
date
1.0    122
Name: count, dtype: int64


In [87]:
# 11.3 Save for Phase 3

ff5.reset_index().to_csv(PROCESSED_DIR / 'phase_2_ff5_factors.csv', index=False)
print('Saved: phase_2_ff5_factors.csv')

Saved: phase_2_ff5_factors.csv


In [88]:
# 11.4 Summary table (the headline result of Phase 2)

cols = ['MKT', 'SMB', 'HML', 'RMW', 'CMA']
n = len(ff5)

summary = pd.DataFrame({
    'mean_monthly_%': ff5[cols].mean() * 100,
    'ann_mean_%': ff5[cols].mean() * 12 * 100,
    'ann_vol_%': ff5[cols].std() * np.sqrt(12) * 100,
    't_stat': ff5[cols].mean() / (ff5[cols].std() / np.sqrt(n)),
})
print(summary.round(2))

print(ff5[cols].corr().round(2))

     mean_monthly_%  ann_mean_%  ann_vol_%  t_stat
MKT            0.70        8.41      17.04    1.58
SMB            0.23        2.80      11.33    0.79
HML            0.53        6.38      11.61    1.76
RMW           -0.15       -1.74       8.79   -0.64
CMA            0.10        1.15       7.31    0.50
      MKT   SMB   HML   RMW   CMA
MKT  1.00  0.37  0.40 -0.34 -0.22
SMB  0.37  1.00  0.35 -0.33 -0.10
HML  0.40  0.35  1.00 -0.66  0.31
RMW -0.34 -0.33 -0.66  1.00 -0.10
CMA -0.22 -0.10  0.31 -0.10  1.00


In [89]:
# ==========================================================
# STEP 12: Save cleaned inputs for Phase 3 test portfolios
# ==========================================================

# Monthly panel of universe members (total returns from Phase 1)
final.to_csv(PROCESSED_DIR / 'phase_2_monthly_panel_clean.csv', index=False)

# Formation snapshot after all Phase 2 filters (Steps 3.6-3.7b, 4.1)
# op is already NaN for financials; SIZE and size_bp included
formation.to_csv(PROCESSED_DIR / 'phase_2_formation.csv', index=False)

print('Saved: phase_2_monthly_panel_clean.csv, phase_2_formation.csv')

Saved: phase_2_monthly_panel_clean.csv, phase_2_formation.csv


| Threshold | Status |
|---|---|
| \|t\| > 2 (5% level) | The long-standing convention in empirical finance, including FF-style papers |
| \|t\| > 3 | Proposed by Harvey, Liu & Zhu (2016) for **new** factors. So many factors have been tested that some pass t > 2 by luck (data mining) |

## Phase 2 Results: FF5 Factor Premia (Jul 2016 – <last month from 11.2>, <N> months)

| Factor | Mean (ann. %) | Vol (ann. %) | t-stat | Significant (\|t\| > 2)? |
|---|---:|---:|---:|:---:|
| MKT | 8.41 | 17.04 | 1.58 | ❌ |
| SMB | 2.80 | 11.33 | 0.79 | ❌ |
| HML | 6.38 | 11.61 | 1.76 | ❌ (significant at 10%) |
| RMW | −1.74 | 8.79 | −0.64 | ❌ |
| CMA | 1.15 | 7.31 | 0.50 | ❌ |

### Effect of removing survivorship bias (Phase 1 v1 → v2)

| Factor | v1: today's Nifty 500 (ann. %, t) | v2: point-in-time universe (ann. %, t) |
|---|---|---|
| MKT | 5.36 (0.96) | 8.41 (1.58) |
| SMB | 15.58 (4.02) | 2.80 (0.79) |
| HML | 10.09 (2.21) | 6.38 (1.76) |
| RMW | −2.84 (−0.78) | −1.74 (−0.64) |
| CMA | 1.02 (0.36) | 1.15 (0.50) |

v1 factors are superseded. Their SMB and HML were inflated by look-ahead: only firms that survived and grew into today's index were included.

### Interpretation

- **Sample:** about 10 years of monthly data, which is short for factor inference. With about 11–12% volatility, a factor needs roughly 7% a year to reach t = 2.

- **MKT (8.4%, t = 1.58):** positive but not significant. The market return now matches the Nifty 500 TRI (Phase 1 check: CAGR 14.1% vs 13.9%). The excess return is modest because Indian Rf is high (about 6% a year) and volatility is high. The COVID crash and the 2024–25 correction both fall inside the window.

- **SMB (2.8%, t = 0.79):** no significant size premium. This agrees with survivorship-free benchmarks: the (Midcap 150 + Smallcap 250)/2 − Nifty 100 spread earned a similar amount, and IIMA's SMB is close to zero or negative. "Small" here means the smaller half of the top 500, which is mostly mid- and small-caps.

- **HML (6.4%, t = 1.76):** the largest characteristic premium, significant at 10% but not at 5%. It reflects the value/PSU re-rating of 2021–24 (PSU banks, energy, metals), so it is driven by one regime.

- **RMW (−1.7%, t = −0.64):** not different from zero. High-profitability "quality" stocks lagged while cyclicals and PSUs rallied. Financials are excluded from this sort.

- **CMA (1.2%, t = 0.50):** essentially zero, so there is no investment premium in this window.

### Factor correlations

- **HML–RMW = −0.66:** in India, value stocks are largely low-profitability firms (PSUs, metals, energy), so HML and RMW carry overlapping information with opposite signs.
- **HML–CMA = +0.31:** positive, as in FF (2015), but weaker than in the US.
- **MKT** correlates 0.37–0.40 with SMB and HML, so part of these premia is market exposure.
- Whether any factor is **redundant** (e.g. HML given MKT, RMW and CMA) is tested in Phase 3 with spanning regressions.

### Overall

- No factor passes \|t\| > 2 over this sample. That reflects the short sample and high volatility, not proof that the premia are absent.
- The point-in-time universe removed the spurious size premium and about a third of the value premium.

### Technical caveats

- These t-stats assume independent months; Phase 3 uses **Newey-West** standard errors (6 lags).
- Universe = top 500 by full market cap, value-weighted with full (not free-float) market cap.
- Rf is the FRED 3-month interbank rate (proxy for the 91-day T-bill).
- Delisting returns exist only for mergers into a listed stock.